# Archived experiment source: V6_DFGAN_StyleGAN2ADA_pipeline_clean.ipynb
All historical cell outputs were cleared for repository publication. The original SHA-256 is recorded in docs/SOURCE_MANIFEST.json.
WARNING: direct Run All may train models, download dependencies, or create files. Use the master orchestrator first. The public folder excludes copyrighted source images and model weights.


# EndekGAN-T2I — V6 External Baselines (CUDA, Project Terpusat)

**Seluruh eksperimen di:** `<PROJECT_ROOT>`

Notebook V6 menghasilkan checkpoint, output StyleGAN2-ADA / DF-GAN–MiniLM, grafik, FID/KID, dan rekaman konfigurasi di subfolder `EndekGAN_V6_EXTERNAL_BASELINES`. Data sumber tetap di folder saudara `EndekGAN_Dataset`. V5 tetap di `EndekGAN_IEEE_V5_OUTPUT`, tidak ditimpa. Cache model disimpan di `MODEL_CACHE`.

**CUDA STRICT:** notebook menghentikan training apabila PyTorch CUDA tidak tersedia. GPU yang sudah dideteksi di komputer penelitian adalah RTX 3070 Ti Laptop GPU. StyleGAN2-ADA resmi masih membutuhkan `nvcc` dan C++ Build Tools, berbeda dari PyTorch CUDA biasa. DF-GAN–MiniLM memakai arsitektur DF-GAN resmi dengan *text embedding* MiniLM adaptasi, bukan model asli dengan DAMSM.

Baseline eksternal **belum dilatih** oleh pembuatan notebook ini. Jangan menafsirkan status `BLOCKED` sebagai metrik/hasil eksperimen. 

## 1 · Mode CUDA STRICT dan konfigurasi

Notebook ini **menolak training CPU**. Pilih kernel VS Code dengan `torch` build CUDA. Jika StyleGAN2-ADA tidak menemukan NVCC, baseline tersebut diberi status blocked; DF-GAN dapat tetap berjalan di CUDA. Status blocked **bukan** hasil eksperimen dan tidak boleh dilaporkan sebagai baseline.


In [ ]:
import sys
import torch

print("Python:", sys.executable)
print("PyTorch:", torch.__version__)
print("CUDA:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
from pathlib import Path
import os, sys, json, csv, zipfile, subprocess, shutil, random, time, hashlib, platform, warnings
import numpy as np, pandas as pd
from PIL import Image

# Seluruh notebook, repositori eksperimen, checkpoint, log, CSV, dan gambar
# menggunakan folder penelitian permanen; tidak bergantung pada Path.cwd().
ROOT = Path(os.environ.get('ENDEKGAN_PROJECT_ROOT', '../EndekGAN-T2I'))
DATASET_ROOT = Path(os.environ.get('ENDEKGAN_DATASET_ROOT', str(ROOT.parent / 'EndekGAN_Dataset')))
V5_OUTPUT = ROOT / 'EndekGAN_IEEE_V5_OUTPUT'  # hasil V5 tetap utuh
V5_ZIP = V5_OUTPUT / 'EndekGAN_IEEE_V5_OUTPUT.zip'
OUTPUT = ROOT / 'EndekGAN_V6_EXTERNAL_BASELINES'
MODEL_CACHE = ROOT / 'MODEL_CACHE'
REPORTS = ROOT / 'PROJECT_REPORTS'
if not ROOT.is_dir():
    raise FileNotFoundError(f'Folder proyek tidak ada: {ROOT}')
for folder in (OUTPUT, MODEL_CACHE, REPORTS):
    folder.mkdir(parents=True, exist_ok=True)
# Cache model HuggingFace dan PyTorch juga tetap berada di folder penelitian.
os.environ['HF_HOME'] = str(MODEL_CACHE / 'huggingface')
os.environ['TORCH_HOME'] = str(MODEL_CACHE / 'torch')
os.environ['XDG_CACHE_HOME'] = str(MODEL_CACHE)
print('DATASET_ROOT:', DATASET_ROOT, '| ada:', DATASET_ROOT.is_dir())
print('V5_OUTPUT:', V5_OUTPUT, '| ada:', V5_OUTPUT.is_dir())
RUN_TRAINING = False       # True: latihan sesungguhnya; False: hanya validasi input
RUN_STYLEGAN2_ADA = False
RUN_DFGAN_ADAPTED = False
RUN_METRICS = False
SEEDS = (42, 123, 2025)
RESOLUTION = 256
N_FAKE_PER_CLASS = 128
# Anggaran V5 ~ 288 images/epoch x 400 epochs = 115,200 image presentations.
ADA_KIMG = 115             # 115,000 images, sesuai tingkat exposure V5, bukan jumlah epoch persis.
ADA_BATCH = 4
DF_EPOCHS = 400
DF_BATCH = 4
DF_NF = 32
DF_Z = 100
DF_VAL_EVERY = 20
DF_MIN_EPOCH = 200
DF_PATIENCE = 80
DF_KID_MIN_DELTA = 0.002
DF_MAGP_COEFF = 1.0       # MAGP bawaan formula official: 2*mean(||grad||^6)
CPU_DF_ALLOWED = False
REQUIRE_CUDA_FOR_TRAINING = True
AUTO_INSTALL_MISSING = True
TRANSFORMER_ID = 'sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2'
GIT_URL_ADA = 'https://github.com/NVlabs/stylegan2-ada-pytorch.git'
GIT_URL_DF = 'https://github.com/tobran/DF-GAN.git'
print('Notebook:', ROOT, '\nV5 ZIP:', V5_ZIP, 'exists:', V5_ZIP.exists())
print('Output:', OUTPUT)

## 2 · Pemeriksaan PyTorch, CUDA, Git, dan dependensi

Tidak menginstal ulang PyTorch secara otomatis agar tidak mengganti versi CUDA milik pengguna. Jika GPU NVIDIA ada tetapi `torch.cuda.is_available()` False, gunakan kernel VS Code yang memiliki PyTorch versi CUDA.
**Choose a Python 3 kernel with CUDA-enabled PyTorch; verify torch.cuda.is_available() before training.**


In [ ]:
import importlib.util
if AUTO_INSTALL_MISSING:
    pkgs={'pandas':'pandas','PIL':'pillow','scipy':'scipy','torchmetrics':'torchmetrics[image]',
          'torch_fidelity':'torch-fidelity','sentence_transformers':'sentence-transformers',
          'yaml':'pyyaml','click':'click','ninja':'ninja','pyspng':'pyspng',
          'imageio_ffmpeg':'imageio-ffmpeg','tqdm':'tqdm'}
    missing=[package for mod,package in pkgs.items() if importlib.util.find_spec(mod) is None]
    if missing:
        print('Installing missing:',missing)
        subprocess.run([sys.executable,'-m','pip','install',*missing],check=True)
import torch, torchvision
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
import torch.nn.functional as F
from torchvision.transforms import functional as TF
print('Python:',sys.version.split()[0],'Torch:',torch.__version__,'Torchvision:',torchvision.__version__)
print('CUDA:',torch.cuda.is_available(), 'GPU:',torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE')
print('Git:',shutil.which('git'), 'NVCC:',shutil.which('nvcc'))
if RUN_TRAINING and REQUIRE_CUDA_FOR_TRAINING:
    if not torch.cuda.is_available():
        raise RuntimeError('CUDA STRICT: PyTorch ini tidak mendukung CUDA. Di komputer ditemukan torch CPU-only; pilih kernel Python dari environment .venv yang sudah diinstal PyTorch +cu128. Training tidak akan menggunakan CPU.')
    device=torch.device('cuda:0')
    # Verifikasi eksekusi operasi nyata pada GPU, bukan hanya driver NVIDIA terdeteksi.
    a=torch.randn(64,64,device=device); b=a @ a; torch.cuda.synchronize()
    print('CUDA STRICT PASSED:',torch.cuda.get_device_name(0), '| torch:',torch.__version__, '| compiled CUDA:',torch.version.cuda,
          '| checked matrix multiply:',float(b[0,0]), '| GPU VRAM total MiB:',torch.cuda.get_device_properties(0).total_memory//(1024**2))
    del a,b
    torch.cuda.empty_cache()
if RUN_TRAINING and RUN_STYLEGAN2_ADA and shutil.which('nvcc') is None:
    print('PERINGATAN: StyleGAN2-ADA official akan BLOCKED_NO_NVCC; DF-GAN CUDA tetap dapat berjalan. Pastikan instal CUDA Toolkit (nvcc) dan C++ build tools sebelum StyleGAN2-ADA.')

if RUN_TRAINING and not V5_ZIP.is_file():
    raise FileNotFoundError(f'Arsip V5 tidak ditemukan: {V5_ZIP}. Folder dan data V5 tidak boleh diganti dengan data dummy.')
status={'StyleGAN2-ADA':'NOT_STARTED','DF-GAN–MiniLM (adapted)':'NOT_STARTED'}
(OUTPUT/'environment.json').write_text(json.dumps({'torch':torch.__version__,'python':sys.version,
  'cuda':torch.cuda.is_available(),'gpu':torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
  'zip':str(V5_ZIP),'project_root':str(ROOT),'output_root':str(OUTPUT),'model_cache':str(MODEL_CACHE),'seeds':list(SEEDS)},indent=2),encoding='utf-8')

## 3 · Pulihkan data *fixed split* V5 dan audit kesesuaian

Input hanya berisi train/val/test dan CSV yang memang tersedia di ZIP. Tidak ada *re-splitting*, tidak ada penggunaan test untuk training. Path Windows `image_path` dalam CSV historis tidak digunakan karena gambar PNG sudah diekspor pada V5.

In [ ]:
if not V5_ZIP.is_file():
    raise FileNotFoundError(V5_ZIP)
DATA = OUTPUT/'private_v5_fixed_data'
DATA.mkdir(exist_ok=True)
with zipfile.ZipFile(V5_ZIP) as z:
    for split in ('train','val','test'):
        csv_name=f'fixed_split/{split}_split.csv'
        frame=pd.read_csv(z.open(csv_name))
        dst=DATA/split;dst.mkdir(exist_ok=True)
        manifest=[]
        for j,row in frame.reset_index(drop=True).iterrows():
            filename=f'{j:05d}.png'
            src=f'E6_StyleGAN2ADA_export/{split}/{filename}'
            if src not in z.namelist():raise RuntimeError(f'V5 ZIP tidak memuat gambar {src}; berhenti agar tidak ada mismatch.')
            output_file=dst/filename
            if not output_file.exists():
                with z.open(src) as fi, open(output_file,'wb') as fo:shutil.copyfileobj(fi,fo)
            with Image.open(output_file) as img:
                assert img.size==(256,256) and img.mode=='RGB', f'Invalid {output_file}'
            manifest.append(dict(file=filename,label=str(row['label']),class_idx=int(row['class_idx']),
                                 prompt=str(row['prompt']),original_filename=str(row['filename']),
                                 md5=str(row['md5'])))
        table=pd.DataFrame(manifest)
        table.to_csv(DATA/f'{split}_aligned.csv',index=False)
        # Label format official NVIDIA for folder dataset.
        (dst/'dataset.json').write_text(json.dumps({'labels':[[t['file'],t['class_idx']] for t in manifest]},indent=2),encoding='utf-8')
        print(split,len(manifest),table.label.value_counts().to_dict())
assert [len(pd.read_csv(DATA/f'{s}_aligned.csv')) for s in ('train','val','test')]==[296,37,37]
train=pd.read_csv(DATA/'train_aligned.csv');val=pd.read_csv(DATA/'val_aligned.csv');test=pd.read_csv(DATA/'test_aligned.csv')
assert set(train['md5']).isdisjoint(set(val['md5'])) and set(train['md5']).isdisjoint(set(test['md5']))
assert set(val['md5']).isdisjoint(set(test['md5']))
CLASS_IDS=sorted(train.class_idx.unique().tolist()); assert CLASS_IDS==[0,1,2,3]
assert train.class_idx.nunique()==4
print('FIXED SPLIT VALIDATED; no leakage; source V5 ZIP:',V5_ZIP.name)

## 4 · Ambil kode arsitektur resmi (Git, otomatis)

Repositori asli diunduh ke folder output lokal. Ini bukan model buatan yang diberi nama baseline. Commit SHA disimpan. Untuk perlindungan integritas, kegagalan download menghasilkan error—bukan *fallback*.

In [ ]:
def get_repo(url,path):
    if not (path/'.git').exists():
        if shutil.which('git') is None:raise RuntimeError('Git belum terpasang: install Git for Windows terlebih dahulu.')
        subprocess.run(['git','clone','--depth','1',url,str(path)],check=True)
    sha=subprocess.run(['git','rev-parse','HEAD'],cwd=str(path),capture_output=True,text=True,check=True).stdout.strip()
    return sha
EXT=OUTPUT/'external_sources';EXT.mkdir(exist_ok=True)
ADA_REPO=EXT/'stylegan2-ada-pytorch';DF_REPO=EXT/'DF-GAN'
if RUN_TRAINING:
    if RUN_STYLEGAN2_ADA and torch.cuda.is_available():
        ada_sha=get_repo(GIT_URL_ADA,ADA_REPO)
        print('Official ADA commit:',ada_sha)
    if RUN_DFGAN_ADAPTED:
        df_sha=get_repo(GIT_URL_DF,DF_REPO)
        print('Official DF-GAN commit:',df_sha)
    (OUTPUT/'source_commits.json').write_text(json.dumps({'stylegan2_ada':locals().get('ada_sha'),
                   'dfgan':locals().get('df_sha')},indent=2),encoding='utf-8')
else:
    print('Preflight only; repos not downloaded')

## 5 · Baseline A: StyleGAN2-ADA **official**, class-conditioned, 3 seed

Kode menjalankan `train.py` NVIDIA, bukan hanya dataset export. CUDA+NVCC+build tools diperlukan. Training official menggunakan jumlah kimg, bukan epoch. `--metrics=none` agar tidak menghitung FID terhadap **train**; evaluasi held-out test dilakukan oleh notebook ini setelah training. **Tidak memakai pretrained FFHQ secara diam-diam**.

Jika tidak ada CUDA, status menjadi `BLOCKED_NO_CUDA` secara eksplisit; tidak diganti dengan imitasi CPU.

In [ ]:
def select_best_ada_snapshot(candidates,seed):
    """Validation-only post-training checkpoint selection (same fixed labels/noise)."""
    sys.path.insert(0,str(ADA_REPO))
    import legacy
    from torchmetrics.image.kid import KernelInceptionDistance
    device=torch.device('cuda')
    val_df=pd.read_csv(DATA/'val_aligned.csv')
    rng=np.random.RandomState(73991)
    best=float('inf');selected=None;log=[]
    real=[DATA/'val'/f'{i:05d}.png' for i in range(len(val_df))]
    # Freeze same random z vector per model snapshot for fair selection.
    for candidate in candidates:
        with open(candidate,'rb') as fd:
            G=legacy.load_network_pkl(fd)['G_ema'].to(device).eval()
        if G.c_dim!=4:raise RuntimeError('Unexpected class labels in official ADA model')
        metric=KernelInceptionDistance(feature=2048,normalize=False,subset_size=10,subsets=20).to(device)
        z=np.random.RandomState(73991).randn(len(real),G.z_dim)
        for start in range(0,len(real),4):
            ids=list(range(start,min(start+4,len(real))))
            orig=torch.stack([torch.from_numpy(np.array(Image.open(real[i]).convert('RGB'),dtype=np.uint8).copy()).permute(2,0,1) for i in ids]).to(device)
            cz=torch.tensor(z[ids],dtype=torch.float32,device=device)
            cond=torch.zeros(len(ids),4,device=device)
            for j,idx in enumerate(ids):cond[j,int(val_df.iloc[idx].class_idx)]=1.0
            with torch.no_grad():fake=G(cz,cond,noise_mode='const')
            fake=(fake*127.5+128).clamp(0,255).byte()
            metric.update(orig,real=True);metric.update(fake,real=False)
        kid=float(metric.compute()[0].cpu())
        log.append({'checkpoint':str(candidate),'validation_KID':kid})
        print('ADA seed',seed,'snapshot',candidate.name,'val KID',f'{kid:.5f}')
        if kid<best:best=kid;selected=candidate
        del G,metric
        torch.cuda.empty_cache()
    pd.DataFrame(log).to_csv(OUTPUT/'training_stylegan2_ada'/f'seed_{seed}'/'validation_snapshot_KID.csv',index=False)
    return selected,best

def ada_run(seed):
    outdir=OUTPUT/'training_stylegan2_ada'/f'seed_{seed}'
    outdir.mkdir(parents=True,exist_ok=True)
    done=outdir/'TRAINING_COMPLETE.json'
    if done.exists():
        return json.loads(done.read_text(encoding='utf-8'))
    cmd=[sys.executable,'train.py',f'--outdir={outdir}',f'--data={DATA/"train"}',
         '--gpus=1','--cond=1','--cfg=auto','--aug=ada','--metrics=none',
         '--mirror=0',f'--kimg={ADA_KIMG}',f'--batch={ADA_BATCH}',
         f'--seed={seed}','--snap=10','--workers=0','--nobench=True']
    log=outdir/'training_console.log'
    print('Official StyleGAN2-ADA seed',seed,'\n',' '.join(map(str,cmd)))
    with open(log,'w',encoding='utf-8') as f:
        proc=subprocess.run(cmd,cwd=str(ADA_REPO),stdout=f,stderr=subprocess.STDOUT)
    if proc.returncode:
        raise RuntimeError(f'StyleGAN2-ADA seed {seed} failed (exit {proc.returncode}). Lihat {log}; periksa CUDA/NVCC/Visual Studio/VRAM. Tidak menandai selesai.')
    candidates=sorted(outdir.rglob('network-snapshot-*.pkl'))
    if not candidates:raise RuntimeError('Training returned success but no network snapshot was found')
    selected,best_val=select_best_ada_snapshot(candidates,seed)
    rec={'seed':seed,'status':'COMPLETE','checkpoint':str(selected),'best_validation_KID':best_val,
         'checkpoint_selection':'validation KID among official snapshots',
         'kimg':ADA_KIMG,'condition':'four-class','source':'NVlabs official'}
    done.write_text(json.dumps(rec,indent=2),encoding='utf-8');return rec

ada_runs=[]
if RUN_TRAINING and RUN_STYLEGAN2_ADA:
    if not torch.cuda.is_available():
        status['StyleGAN2-ADA']='BLOCKED_NO_CUDA'
        print('OFFICIAL STYLEGAN2-ADA NOT TRAINED: CUDA unavailable. CPU implementation is NOT equivalent.')
    elif shutil.which('nvcc') is None:
        status['StyleGAN2-ADA']='BLOCKED_NO_NVCC'
        print('CUDA torch detected but nvcc missing. Install matching CUDA toolkit and C++ build tools.')
    else:
        for seed in SEEDS:
            try:
                rec=ada_run(seed);ada_runs.append(rec);print('Finished:',seed)
            except Exception as exc:
                status['StyleGAN2-ADA']='ERROR';print(type(exc).__name__,exc)
                break
        if len(ada_runs)==len(SEEDS):status['StyleGAN2-ADA']='COMPLETE'
print('StyleGAN status:',status['StyleGAN2-ADA'])

## 6 · Baseline B: **DF-GAN–MiniLM (adapted)**, official NetG/NetD/NetC + MAGP

Arsitektur DF-GAN diimpor dari file asli `code/models/GAN.py` (`NetG`, `NetD`, `NetC`), dengan *matching-aware gradient penalty*, *mismatched text negatives*, dan loss adversarial berdasarkan proyek asli. **Perbedaan utama:** caption encoder DAMSM asli diganti encoder multilingual MiniLM 384-D agar semua kondisi teks Endek dalam Bahasa Indonesia tetap dapat dipakai. Jelas dilabeli **adapted**, tidak boleh ditulis sebagai reproduksi official DF-GAN tanpa modifikasi.

Training menggunakan weighted sampling berdasarkan kelas (seperti V5), fixed split, maksimal 400 epoch, checkpoint lengkap untuk resume, dan pemilihan *best epoch* dengan KID validation saja. Bila CPU dipilih, eksperimen tetap benar-benar dilatih tetapi lambat.

In [ ]:
from tqdm.auto import tqdm
from copy import deepcopy
# Offline embeddings precomputed ONCE; never use hash fallback.
EMBED_DIR=OUTPUT/'embeddings';EMBED_DIR.mkdir(exist_ok=True)
if RUN_TRAINING and RUN_DFGAN_ADAPTED:
    from sentence_transformers import SentenceTransformer
    embedding_files={s:EMBED_DIR/f'{s}_MiniLM384.npy' for s in ('train','val','test')}
    if not all(p.exists() for p in embedding_files.values()):
        encoder=SentenceTransformer(TRANSFORMER_ID,device='cpu')
        for split in ('train','val','test'):
            df=pd.read_csv(DATA/f'{split}_aligned.csv')
            a=encoder.encode(df.prompt.astype(str).tolist(),batch_size=16,normalize_embeddings=True,
                             show_progress_bar=True,convert_to_numpy=True)
            assert a.shape==(len(df),384)
            np.save(embedding_files[split],a.astype(np.float32))
        del encoder
    else:print('Existing multilingual embeddings reused')
    sys.path.insert(0,str(DF_REPO/'code'))
    from models.GAN import NetG, NetD, NetC
    print('DF-GAN official architecture imported; MiniLM vectors: 384D')
else:
    print('DF-GAN embedding/training disabled in current configuration')

class EndekPairs(Dataset):
    def __init__(self, split):
        self.split=split;self.df=pd.read_csv(DATA/f'{split}_aligned.csv');
        self.emb=torch.from_numpy(np.load(EMBED_DIR/f'{split}_MiniLM384.npy'))
        assert len(self.df)==len(self.emb)
    def __len__(self):return len(self.df)
    def __getitem__(self,i):
        name=self.df.iloc[i]['file']
        with Image.open(DATA/self.split/name) as img:
            x=TF.to_tensor(img.convert('RGB'))*2-1
        return x,self.emb[i],int(self.df.iloc[i]['class_idx'])

def kid_eval(netG,ds,seed,device,subsets=20):
    from torchmetrics.image.kid import KernelInceptionDistance
    subset=min(10,len(ds)); metric=KernelInceptionDistance(feature=2048,normalize=False,
                         subset_size=subset,subsets=subsets).to(device)
    g=torch.Generator(device='cpu');g.manual_seed(seed)
    z_all=torch.randn(len(ds),DF_Z,generator=g)
    netG.eval()
    loader=DataLoader(ds,batch_size=4,shuffle=False,num_workers=0)
    offset=0
    for real,emb,_ in loader:
        count=len(real); z=z_all[offset:offset+count].to(device);offset+=count
        with torch.no_grad():
            fake=netG(z,emb.to(device))
        real_uint=((real+1)*127.5).clamp(0,255).to(torch.uint8).to(device)
        fake_uint=((fake+1)*127.5).clamp(0,255).to(torch.uint8)
        metric.update(real_uint,real=True);metric.update(fake_uint,real=False)
    result=metric.compute()[0].item()
    netG.train()
    return float(result)

### Prosedur training DF-GAN (resume-safe)

Optimizer G & D Adam sesuai DF-GAN, *gradient penalty* MAGP pada image+text, dan mismatch negatives. Tidak ada penggantian model secara diam-diam jika gagal. Pengukuran KID dari validation kecil bersifat eksploratif.

In [ ]:
def df_train(seed):
    device=torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
    if device.type=='cpu' and not CPU_DF_ALLOWED:
        raise RuntimeError('CPU_DF_ALLOWED=False and CUDA missing')
    random.seed(seed);np.random.seed(seed);torch.manual_seed(seed)
    if device.type=='cuda':torch.cuda.manual_seed_all(seed)
    folder=OUTPUT/'training_dfgan_minilm_adapted'/f'seed_{seed}'
    folder.mkdir(parents=True,exist_ok=True)
    done=folder/'TRAINING_COMPLETE.json'
    if done.exists():return json.loads(done.read_text(encoding='utf-8'))
    modelG=NetG(DF_NF,DF_Z,384,RESOLUTION,3).to(device)
    modelD=NetD(DF_NF,RESOLUTION,3).to(device)
    modelC=NetC(DF_NF,384).to(device)
    optG=torch.optim.Adam(modelG.parameters(),lr=1e-4,betas=(0.0,0.9))
    optD=torch.optim.Adam(list(modelD.parameters())+list(modelC.parameters()),lr=4e-4,betas=(0.0,0.9))
    tr=EndekPairs('train');va=EndekPairs('val')
    counts=np.bincount(tr.df.class_idx.to_numpy(),minlength=4)
    weights=(1.0/counts[tr.df.class_idx.to_numpy()]).tolist()
    samples=(len(tr)//DF_BATCH)*DF_BATCH
    ckpt=folder/'last_resume.pth';bestp=folder/'best_validation_kid.pth'
    start=1; best_score=float('inf');best_epoch=None;history=[]
    if ckpt.exists():
        old=torch.load(ckpt,map_location=device,weights_only=False)
        if old['seed']!=seed:raise RuntimeError('Checkpoint seed mismatch')
        modelG.load_state_dict(old['G']);modelD.load_state_dict(old['D']);modelC.load_state_dict(old['C'])
        optG.load_state_dict(old['optG']);optD.load_state_dict(old['optD'])
        start=old['epoch']+1;best_score=old['best_score'];best_epoch=old['best_epoch']
        history=old['history'];print('RESUME DF-GAN seed',seed,'epoch',start)
    for epoch in range(start,DF_EPOCHS+1):
        sampler=WeightedRandomSampler(weights,num_samples=samples,replacement=True,
                          generator=torch.Generator().manual_seed(seed*100000+epoch))
        loader=DataLoader(tr,batch_size=DF_BATCH,sampler=sampler,num_workers=0,drop_last=True)
        loss_gs=[];loss_ds=[]
        modelG.train();modelD.train();modelC.train()
        for real,txt,_ in tqdm(loader,desc=f'DF-GAN seed {seed} epoch {epoch}/{DF_EPOCHS}',leave=False):
            real=real.to(device).detach().requires_grad_(True)
            txt=txt.to(device).detach().requires_grad_(True)
            # Real / mismatched real / fake. Matches CVPR DF-GAN training formulation.
            feat_real=modelD(real)
            pred_real=modelC(feat_real,txt)
            misfeat=torch.cat([feat_real[1:],feat_real[:1]],dim=0)
            d_real=F.relu(1-pred_real).mean()
            d_mis=F.relu(1+modelC(misfeat,txt)).mean()
            z=torch.randn(len(real),DF_Z,device=device)
            fake=modelG(z,txt.detach())
            d_fake=F.relu(1+modelC(modelD(fake.detach()),txt)).mean()
            grads=torch.autograd.grad(pred_real,[real,txt],grad_outputs=torch.ones_like(pred_real),
                                      retain_graph=True,create_graph=True)
            norm=torch.cat([grads[0].flatten(1),grads[1].flatten(1)],dim=1).norm(dim=1)
            magp=2.0*torch.mean(norm.pow(6))
            d_loss=d_real+0.5*(d_fake+d_mis)+DF_MAGP_COEFF*magp
            optD.zero_grad(set_to_none=True);d_loss.backward();optD.step()
            for p in list(modelD.parameters())+list(modelC.parameters()):p.requires_grad_(False)
            g_loss=-modelC(modelD(fake),txt.detach()).mean()
            optG.zero_grad(set_to_none=True);g_loss.backward();optG.step()
            for p in list(modelD.parameters())+list(modelC.parameters()):p.requires_grad_(True)
            loss_gs.append(float(g_loss.detach()));loss_ds.append(float(d_loss.detach()))
        row={'epoch':epoch,'generator_loss':np.mean(loss_gs),'discriminator_loss':np.mean(loss_ds),
             'validation_KID':np.nan}
        should_check=(epoch%DF_VAL_EVERY==0 or epoch==DF_EPOCHS)
        if should_check:
            score=kid_eval(modelG,va,seed=987654,device=device,subsets=20)
            row['validation_KID']=score
            print(f'DF-GAN seed={seed} epoch={epoch} validation KID={score:.5f}; current best={best_score:.5f}')
            if score+DF_KID_MIN_DELTA <best_score:
                best_score=score;best_epoch=epoch
                torch.save({'G':modelG.state_dict(),'seed':seed,'epoch':epoch,'validation_KID':score,
                            'description':'DF-GAN architecture + MiniLM384 conditioning'},bestp)
        history.append(row)
        # Save full optimizer state for exact epoch resume (within same environment).
        torch.save({'G':modelG.state_dict(),'D':modelD.state_dict(),'C':modelC.state_dict(),
          'optG':optG.state_dict(),'optD':optD.state_dict(),'epoch':epoch,'seed':seed,
          'best_score':best_score,'best_epoch':best_epoch,'history':history},ckpt)
        pd.DataFrame(history).to_csv(folder/'training_log.csv',index=False)
        if epoch >=DF_MIN_EPOCH and best_epoch is not None and epoch-best_epoch>=DF_PATIENCE:
            print('VALIDATION EARLY STOPPING at',epoch,'best epoch',best_epoch);break
    if not bestp.exists():raise RuntimeError('Validation best checkpoint not created')
    rec={'seed':seed,'status':'COMPLETE','checkpoint':str(bestp),'best_epoch':best_epoch,
         'last_epoch':epoch,'best_validation_KID':best_score,
         'text_encoder':TRANSFORMER_ID,'training':'DF-GAN official architecture adapted to 384-D MiniLM'}
    done.write_text(json.dumps(rec,indent=2),encoding='utf-8');return rec

df_runs=[]
if RUN_TRAINING and RUN_DFGAN_ADAPTED:
    for seed in SEEDS:
        try:
            rec=df_train(seed);df_runs.append(rec)
            print('DF-GAN completed seed',seed,'best',rec['best_epoch'])
        except Exception as exc:
            status['DF-GAN–MiniLM (adapted)']='ERROR'
            print('ERROR DF-GAN seed',seed,repr(exc));raise
    if len(df_runs)==len(SEEDS):status['DF-GAN–MiniLM (adapted)']='COMPLETE'
print('DF-GAN status:',status['DF-GAN–MiniLM (adapted)'])

## 7 · Generate 128 images per class × 4 classes × 3 seed (TEST evaluation)

Checkpoint dipilih dengan **validation only**. Untuk DF-GAN, prompt diambil dari split TEST dan diulang secara seimbang per kelas; hanya untuk generasi/evaluasi, bukan training. Untuk StyleGAN2-ADA, label kelas menjadi kondisi generator.

In [ ]:
from collections import defaultdict
EVAL_ROOT=OUTPUT/'external_generated_test';EVAL_ROOT.mkdir(exist_ok=True)

def synthesize_df(seed,checkpoint):
    device=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model=NetG(DF_NF,DF_Z,384,RESOLUTION,3).to(device)
    payload=torch.load(checkpoint,map_location=device,weights_only=True)
    model.load_state_dict(payload['G']);model.eval()
    dset=EndekPairs('test');base=EVAL_ROOT/f'DFGAN_MiniLM_seed_{seed}'
    by_class=defaultdict(list)
    for i,c in enumerate(dset.df.class_idx.tolist()):by_class[int(c)].append(i)
    for cls in CLASS_IDS:
        target=base/str(cls);target.mkdir(parents=True,exist_ok=True)
        g=torch.Generator().manual_seed(seed*100+cls)
        prompts=torch.stack([dset.emb[by_class[cls][j%len(by_class[cls])]] for j in range(N_FAKE_PER_CLASS)])
        noise=torch.randn(N_FAKE_PER_CLASS,DF_Z,generator=g)
        for st in range(0,N_FAKE_PER_CLASS,4):
            with torch.no_grad():fake=model(noise[st:st+4].to(device),prompts[st:st+4].to(device))
            imgs=(fake.add(1).mul(127.5).clamp(0,255).byte().permute(0,2,3,1).cpu().numpy())
            for j,img in enumerate(imgs):Image.fromarray(img).save(target/f'{st+j:04d}.png')
    return base

def synthesize_ada(seed,checkpoint):
    import pickle
    sys.path.insert(0,str(ADA_REPO))
    import legacy
    device=torch.device('cuda')
    with open(checkpoint,'rb') as f: gen=legacy.load_network_pkl(f)['G_ema'].to(device).eval()
    if gen.c_dim!=4:raise RuntimeError(f'Expected four class labels, official G.c_dim={gen.c_dim}')
    base=EVAL_ROOT/f'StyleGAN2ADA_seed_{seed}'
    for cls in CLASS_IDS:
        folder=base/str(cls);folder.mkdir(parents=True,exist_ok=True)
        rng=np.random.RandomState(seed*100+cls)
        for st in range(0,N_FAKE_PER_CLASS,4):
            zs=torch.from_numpy(rng.randn(min(4,N_FAKE_PER_CLASS-st),gen.z_dim)).float().to(device)
            c=torch.zeros(len(zs),4,device=device);c[:,cls]=1
            with torch.no_grad(): imgs=gen(zs,c,noise_mode='const')
            imgs=(imgs*127.5+128).clamp(0,255).to(torch.uint8).permute(0,2,3,1).cpu().numpy()
            for j,img in enumerate(imgs):Image.fromarray(img).save(folder/f'{st+j:04d}.png')
    return base

models=[]
if RUN_TRAINING:
    if status['DF-GAN–MiniLM (adapted)']=='COMPLETE':
        for rec in df_runs:models.append(('DFGAN_MiniLM',rec['seed'],synthesize_df(rec['seed'],rec['checkpoint'])))
    if status['StyleGAN2-ADA']=='COMPLETE':
        for rec in ada_runs:models.append(('StyleGAN2ADA',rec['seed'],synthesize_ada(rec['seed'],rec['checkpoint'])))
print('Generated outputs ready:',len(models),'models/seeds')

## 8 · Same held-out FID/KID protocol as V5 (exploratory)

`torchmetrics` Inception FID/KID, real test subset per class (6–14 images), **128 fake per class**. Hindari membandingkan langsung dengan skor publikasi CUB/COCO; sampel real sangat kecil. Output mengandung data per class, macro, dan pooled global (bukan saling dipertukarkan).

In [ ]:
def load_u8(path):
    with Image.open(path) as img:
        x=np.asarray(img.convert('RGB').resize((RESOLUTION,RESOLUTION)),dtype=np.uint8).copy()
    return torch.from_numpy(x).permute(2,0,1)

def evaluate_paths(real_paths,fake_paths,device):
    from torchmetrics.image.fid import FrechetInceptionDistance
    from torchmetrics.image.kid import KernelInceptionDistance
    nreal=len(real_paths); nfake=len(fake_paths)
    if nreal<4 or nfake<4:raise ValueError('Not enough images for Inception metrics')
    fm=FrechetInceptionDistance(feature=2048,normalize=False).to(device)
    km=KernelInceptionDistance(feature=2048,normalize=False,
         subset_size=min(50,nreal,nfake),subsets=50).to(device)
    for files,isreal in ((real_paths,True),(fake_paths,False)):
        for i in range(0,len(files),4 if device.type=='cpu' else 16):
            batch=torch.stack([load_u8(x) for x in files[i:i+(4 if device.type=='cpu' else 16)]]).to(device)
            fm.update(batch,real=isreal);km.update(batch,real=isreal)
    fid=float(fm.compute().cpu());k,sd=km.compute()
    return {'FID':fid,'KID':float(k.cpu()),'KID_subsample_SD':float(sd.cpu()),
            'n_real':nreal,'n_fake':nfake}

metrics=[]
if RUN_TRAINING and RUN_METRICS:
    metric_device=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    for name,seed,path in models:
        for cls in CLASS_IDS:
            real=sorted((DATA/'test').glob('*.png'))
            cls_indexes=test.index[test.class_idx==cls].tolist()
            real=[DATA/'test'/f'{i:05d}.png' for i in cls_indexes]
            fake=sorted((path/str(cls)).glob('*.png'))
            res=evaluate_paths(real,fake,metric_device)
            metrics.append({'method':name,'seed':seed,'class':cls,'scope':'per_class',**res})
        fake=sorted(path.glob('*/*.png'));real=sorted((DATA/'test').glob('*.png'))
        res=evaluate_paths(real,fake,metric_device)
        metrics.append({'method':name,'seed':seed,'class':'all','scope':'pooled_global',**res})
        pd.DataFrame(metrics).to_csv(OUTPUT/'E8_EXTERNAL_FID_KID_PER_CLASS.csv',index=False)
        print(name,seed,'metrics saved')
    per=pd.DataFrame(metrics);per=per[per.scope=='per_class']
    byrun=per.groupby(['method','seed'])[['FID','KID']].mean().reset_index()
    byrun=byrun.rename(columns={'FID':'macro_FID_exploratory','KID':'macro_KID_exploratory'})
    byrun.to_csv(OUTPUT/'E8_EXTERNAL_MACRO_BY_RUN.csv',index=False)
    print(byrun.to_string(index=False))
else:print('No new external model results yet; no zero/placeholder metrics inserted')

## 9 · Hasil perbandingan akhir (V5 + dua baseline eksternal)

Hasil V5 hanya dibaca dari ZIP lama, tidak disentuh. Skor lama dipisahkan jelas sebagai hasil V5; skor eksternal yang gagal atau belum dilatih tetap `missing` (bukan angka buatan).

In [ ]:
with zipfile.ZipFile(V5_ZIP) as z:
    v5=pd.read_csv(z.open('E1_E7_NEW_RESULTS_BY_RUN.csv'))
v5=v5.rename(columns={'run':'model_run','macro_FID_exploratory':'macro_FID',
                      'macro_KID_exploratory':'macro_KID'})
v5['source']='V5_internal'
parts=[v5[['model_run','seed','macro_FID','macro_KID','source']]]
e8=OUTPUT/'E8_EXTERNAL_MACRO_BY_RUN.csv'
if e8.exists():
    ext=pd.read_csv(e8)
    ext['model_run']=ext['method']+'_'+ext.seed.astype(str)
    ext=ext.rename(columns={'macro_FID_exploratory':'macro_FID','macro_KID_exploratory':'macro_KID'})
    ext['source']='V6_external'
    parts.append(ext[['model_run','seed','macro_FID','macro_KID','source']])
combined=pd.concat(parts,ignore_index=True)
combined.to_csv(OUTPUT/'E9_V5_V6_COMPARISON_ACTUAL_ONLY.csv',index=False)
print(combined.to_string(index=False,max_rows=30))
(OUTPUT/'TRAINING_STATUS.json').write_text(json.dumps(status,indent=2),encoding='utf-8')
print('Training status:',status)
if len(parts)==1:
    print('External training metrics not present; do NOT claim baseline training completed.')

## 10 · Instruksi interpretasi untuk artikel IEEE

- Metode eksternal ini adalah **StyleGAN2-ADA (official class-conditioned)** dan **DF-GAN–MiniLM (architecturally adapted)**. Pisahkan keduanya dari 18 konfigurasi V5.
- Training StyleGAN2-ADA menggunakan **115 kimg**, sedangkan DF-GAN sampai 400 epoch dengan early stopping; laporkan *actual processed images/updates* dan waktu, bukan menyatakan anggaran sepenuhnya identik. CUDA diperlukan hanya untuk StyleGAN2-ADA official.
- Prompt dalam V5 adalah **deterministic, not expert-verified**. Untuk klaim *prompt adherence*, diperlukan eksperimen intervensi prompt dan penilaian domain independen.
- FID/KID dengan 37 gambar real test bersifat eksploratif; tidak boleh mengklaim signifikansi dari angka tersebut saja.
- Jangan membagikan isi `private_v5_fixed_data` atau ZIP yang mengandung gambar asli tanpa izin sumber.
- Notebook hanya dianggap **hasil eksperimen** jika ada `TRAINING_COMPLETE.json`, checkpoint, gambar fake, dan CSV evaluasi dengan status benar-benar COMPLETE. Kegagalan dependency/CUDA harus diperbaiki, bukan ditutupi.